<div style="background:linear-gradient(135deg,#4c0519 0%,#be123c 55%,#fb7185 100%);border-radius:18px;padding:32px 30px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
  <div style="font-size:12px;letter-spacing:3px;color:#fecdd3;font-weight:700;text-transform:uppercase">Chapter 157 &middot; Communicating Results &middot; Report 2 of 3</div>
  <div style="font-size:32px;font-weight:900;line-height:1.1;margin:10px 0 6px">Experiment Readout: A/B Test Results</div>
  <div style="font-size:15px;color:#ffe4e6;max-width:760px;line-height:1.6">The decision memo. An experiment readout exists to answer one question, ship it or not, and to show the evidence honestly, uncertainty included. This notebook analyzes the test and writes the decision as a Word document.</div>
</div>

In [ ]:
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
# A clean house style for report-ready figures: no chartjunk, strong titles, muted grid.
mpl.rcParams.update({"figure.dpi":110,"font.size":11,"axes.spines.top":False,"axes.spines.right":False,
    "axes.grid":True,"grid.alpha":0.22,"axes.titleweight":"bold","axes.titlesize":12.5,
    "axes.titlelocation":"left","axes.titlepad":10})
ROSE, INK, MUT, GR, RD = "#be123c", "#1a2138", "#64748b", "#16a34a", "#dc2626"
BASE = "https://raw.githubusercontent.com/johnfisher-ai/Statistics-Data-Science-AI-Visual-Book/main/data/"
fn = "communicating-insights--company-data.xlsx"
def load(sheet):
    try: return pd.read_excel("../../data/" + fn, sheet_name=sheet)
    except FileNotFoundError: return pd.read_excel(BASE + fn, sheet_name=sheet)
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from pathlib import Path
import tempfile
ROSE_DOC, GREY_DOC = RGBColor(0xBE,0x12,0x3C), RGBColor(0x64,0x74,0x8B)
def new_report(title, subtitle):
    doc = Document()
    for s in doc.sections:                       # US Letter, sensible margins
        s.page_width, s.page_height = Inches(8.5), Inches(11)
        s.left_margin = s.right_margin = Inches(1); s.top_margin = s.bottom_margin = Inches(0.9)
    t = doc.add_heading(title, level=0)
    for r in t.runs: r.font.color.rgb = ROSE_DOC
    sp = doc.add_paragraph(subtitle); sp.runs[0].italic = True; sp.runs[0].font.color.rgb = GREY_DOC
    return doc
def h2(doc, text):
    hd = doc.add_heading(text, level=1)
    for r in hd.runs: r.font.color.rgb = ROSE_DOC
def bullets(doc, items):
    for it in items: doc.add_paragraph(it, style="List Bullet")
def df_table(doc, df):                           # df cells should already be display strings
    tb = doc.add_table(rows=1, cols=len(df.columns)); tb.style = "Light Grid Accent 1"
    for j, c in enumerate(df.columns):
        cell = tb.rows[0].cells[j]; cell.text = str(c)
        for r in cell.paragraphs[0].runs: r.bold = True
    for _, row in df.iterrows():
        cells = tb.add_row().cells
        for j, c in enumerate(df.columns): cells[j].text = str(row[c])
    return tb
def add_fig(doc, fig, width=6.3):
    p = Path(tempfile.mkdtemp()) / "fig.png"; fig.savefig(p, dpi=150, bbox_inches="tight"); plt.close(fig)
    doc.add_picture(str(p), width=Inches(width)); doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
def save_report(doc, name):                       # write into the book repo when present, else the cwd (Colab)
    outdir = Path("../../reports") if Path("../../reports").exists() else Path(".")
    outdir.mkdir(exist_ok=True); path = outdir / name; doc.save(path)
    print("wrote", path.resolve()); return path
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

## Step 1 &middot; Analyze the experiment
Two versions of a checkout, split evenly. Compute the conversion rates, the lift, and, crucially, the statistical evidence: a two-proportion test and a confidence interval on the difference.

In [ ]:
ab = load("ABTest")
vis = ab.visitors.values; conv = ab.conversions.values
rA, rB = conv[0]/vis[0], conv[1]/vis[1]
abs_lift, rel_lift = rB-rA, (rB-rA)/rA
z, p = proportions_ztest([conv[1], conv[0]], [vis[1], vis[0]])
ci_low, ci_high = confint_proportions_2indep(conv[1], vis[1], conv[0], vis[0])
print(f"A {rA:.2%}  B {rB:.2%}  | abs lift {abs_lift:+.2%}pts  rel {rel_lift:+.0%}")
print(f"z {z:.2f}  p {p:.4f}  | 95% CI on difference [{ci_low:+.2%}, {ci_high:+.2%}]")
decision = "SHIP the new checkout" if (p < 0.05 and abs_lift > 0) else "DO NOT ship; keep the control"
print("decision:", decision)

## Step 2 &middot; One honest chart
A bar of the two rates with a confidence interval on each. Zero baseline, so the reader sees the true size of the effect, not an inflated one.

In [ ]:
from statsmodels.stats.proportion import proportion_confint
lo = [proportion_confint(conv[i], vis[i])[0]*100 for i in range(2)]
hi = [proportion_confint(conv[i], vis[i])[1]*100 for i in range(2)]
rates = [rA*100, rB*100]
fig_ab, ax = plt.subplots(figsize=(6.4, 3.4))
bars = ax.bar(["A (control)","B (new checkout)"], rates, color=[MUT, ROSE],
              yerr=[np.subtract(rates,lo), np.subtract(hi,rates)], capsize=6)
for b,v in zip(bars, rates): ax.text(b.get_x()+b.get_width()/2, v+0.15, f"{v:.1f}%", ha="center", fontweight="bold")
ax.set_ylim(0, 11); ax.set_ylabel("conversion rate (%)")
ax.set_title(f"B converts {rel_lift:.0%} higher (p = {p:.3f})"); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

## Step 3 &middot; Write the decision memo
Decision first, in bold, then the method, the evidence table, the chart, and an honest caveat. A reader should be able to act on the first line and audit the rest.

In [ ]:
doc = new_report("A/B Test Readout: New Checkout Flow", "Experiment ran 2 weeks  |  Owner: Growth team")

h2(doc, "Decision")
verdict = doc.add_paragraph()
run = verdict.add_run(decision + ".")
run.bold = True; run.font.size = Pt(13)
doc.add_paragraph(f"The new checkout lifted conversion from {rA:.1%} to {rB:.1%}, a {rel_lift:.0%} relative gain. "
                  f"The result is statistically significant (p = {p:.3f}) and the plausible effect stays positive.")

h2(doc, "Method")
doc.add_paragraph(f"Visitors were randomly split between the current checkout (A) and a redesigned flow (B), "
                  f"{vis[0]:,} per arm. Success is a completed purchase. We compare conversion rates with a "
                  f"two-proportion z-test and report a 95% confidence interval on the difference.")

h2(doc, "Results")
res = pd.DataFrame({"Group":["A (control)","B (new checkout)"],
                    "Visitors":[f"{vis[0]:,}", f"{vis[1]:,}"],
                    "Conversions":[f"{conv[0]:,}", f"{conv[1]:,}"],
                    "Rate":[f"{rA:.1%}", f"{rB:.1%}"]})
df_table(doc, res)
add_fig(doc, fig_ab, width=5.4)
doc.add_paragraph(f"Relative lift {rel_lift:+.0%}. Two-proportion z = {z:.2f}, p = {p:.3f}. "
                  f"95% confidence interval on the absolute difference: [{ci_low:+.2%}, {ci_high:+.2%}].")

h2(doc, "Caveats")
bullets(doc, ["The test ran for two weeks; a longer window would confirm the effect holds beyond a novelty bump.",
              "Conversion is the only metric here; monitor refunds and support tickets after rollout.",
              "The interval is comfortably above zero, but the lower bound is modest, so treat the point estimate as optimistic."])

path = save_report(doc, "report-2-ab-test-readout.docx")

### Wrap-up
The experiment genre puts the verdict in the first line and the uncertainty in plain sight: a confidence interval, a p-value, and honest caveats. Notice the report never overclaims, it reports the interval, not just the point estimate, which is exactly the discipline the inference chapters taught.